# 🌊 AI Urban Flood & Traffic Congestion Monitor (Official SYNEXAN)
### 100% Giao diện Gradio gốc ở Local (Tesla T4 GPU + Cloudflare Tunnel Siêu Tốc)
Triển khai tối ưu trên Kaggle GPU (Tesla T4) + Cloudflare Tunnel (Băng thông KHÔNG GIỚI HẠN).
Tự động kết nối và đồng bộ siêu tốc về Web App trên Vercel: https://frontend-flood.vercel.app


In [ ]:
# 1. Cài đặt các thư viện cần thiết tinh gọn & Tải Cloudflare Tunnel
import os, subprocess, urllib.request

# Kaggle đã có sẵn torch, cuda, opencv, pillow, pandas, nest-asyncio
# Cài đặt chuẩn hóa tương thích tuyệt đối cho Gradio 4.44.0 + Ultralytics + FastAPI
!pip install -q --no-cache-dir --prefer-binary "gradio==4.44.0" "pydantic<2.10.0" "huggingface_hub<0.26.0" ultralytics lapx fastapi uvicorn python-multipart

if not os.path.exists("/tmp/cloudflared"):
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)


In [ ]:
# 2. Đồng bộ mã nguồn Backend siêu tốc từ GitHub SYNEXAN/FrontendFlood (--depth 1)
import os, sys

!rm -rf /kaggle/working/FrontendFlood
!git clone --depth 1 -q https://github.com/SYNEXAN/FrontendFlood.git /kaggle/working/FrontendFlood

os.chdir("/kaggle/working/FrontendFlood/backend")
sys.path.append("/kaggle/working/FrontendFlood/backend")
print("✅ Thư mục backend hiện tại:", os.getcwd())


In [ ]:
# 3. Tải song song Trọng số mô hình Deep Learning (ThreadPoolExecutor)
import os, urllib.request, time
from concurrent.futures import ThreadPoolExecutor

os.makedirs("models", exist_ok=True)
models_info = [
    ("models/vietnam_flood_yolov8s_seg.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/vietnam_flood_yolov8s_seg.pt", 20000000),
    ("models/yolov8s_vehicle.pt", "https://huggingface.co/spaces/thietnt/AI-Urban-Flood-Traffic-Monitor/resolve/main/models/yolov8s_vehicle.pt", 20000000)
]

def download_one(path, url, min_size):
    if os.path.exists(path) and os.path.getsize(path) >= min_size:
        print(f"✅ Mô hình {path} đã có sẵn ({os.path.getsize(path)} bytes).")
        return True
    for attempt in range(1, 4):
        try:
            print(f"⬇️ Đang tải {path} (lần {attempt}/3)...")
            if os.path.exists(path):
                os.remove(path)
            urllib.request.urlretrieve(url, path)
            if os.path.getsize(path) >= min_size:
                print(f"✅ Tải thành công {path} ({os.path.getsize(path)} bytes).")
                return True
        except Exception as e:
            print(f"❌ Lỗi tải {path}: {e}")
            time.sleep(1)
    return False

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(download_one, path, url, min_size) for path, url, min_size in models_info]
    for f in futures:
        f.result()

print("✅ Sẵn sàng các file mô hình:")
!ls -lh models/


In [ ]:
# 4. Khởi động Dual-Tunnel (Cloudflare & Gradio Live) & Gradio App Song Song
import subprocess, time, os, re, urllib.request, json, threading, sys, socket

TARGET_WEBHOOKS = [
    "https://frontend-flood.vercel.app/api/backend-url",
    "https://frontendflood.vercel.app/api/backend-url",
    "https://frontendflood.onrender.com/api/backend-url"
]

KERNEL_LOG_ENDPOINT = "https://frontend-flood.vercel.app/api/kernel-log"
BACKEND_DIR = "/kaggle/working/FrontendFlood/backend"

def notify_kernel_log(status, message, details=None):
    try:
        data = json.dumps({"status": status, "message": message, "details": details}).encode("utf-8")
        req = urllib.request.Request(
            KERNEL_LOG_ENDPOINT,
            data=data,
            headers={
                "Content-Type": "application/json",
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
            }
        )
        urllib.request.urlopen(req, timeout=4)
    except Exception:
        pass

!pkill -f cloudflared
!pkill -f "python app.py"

if not os.path.exists("/tmp/cloudflared"):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/tmp/cloudflared")
    os.chmod("/tmp/cloudflared", 0o755)

def check_port_open(port=7860):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(1)
    try:
        s.connect(('127.0.0.1', port))
        s.close()
        return True
    except Exception:
        return False

# Khởi động Cloudflare Tunnel với giao thức HTTP/2 TCP
log_file = open("/tmp/cloudflared.log", "w")
cf_cmd = ["/tmp/cloudflared", "tunnel", "--protocol", "http2", "--url", "http://127.0.0.1:7860"]
cf_proc = subprocess.Popen(cf_cmd, stdout=log_file, stderr=subprocess.STDOUT)

app_proc = None

def dispatch_tunnel_url():
    """Background Worker: Chờ Gradio sẵn sàng trên port 7860 và bắt đường hầm Cloudflare/Gradio"""
    print("⏳ [Dual-Tunnel Hook] Đang đợi Gradio nạp mô hình và mở cổng 7860...")
    notify_kernel_log("running", "GPU Tesla T4 đã kích hoạt! Đang nạp mô hình AI vào RAM...")
    
    # 1. Đợi cổng 7860 mở thực sự
    gradio_ready = False
    for wait_s in range(1, 180):
        time.sleep(1)
        if app_proc and app_proc.poll() is not None and app_proc.returncode != 0:
            print(f"❌ [Gradio Process] app.py đã thoát lỗi với mã: {app_proc.returncode}")
            break
        if check_port_open(7860):
            gradio_ready = True
            print(f"\n🎉 [Gradio Ready] Cổng 7860 đã mở sau {wait_s}s!\n")
            break
        if wait_s % 15 == 0:
            notify_kernel_log("running", f"Đang nạp mô hình Deep Learning vào GPU ({wait_s}s/180s)...")
    
    # 2. Tìm Cloudflare Tunnel URL hoặc Gradio Live URL
    public_url = None
    for elapsed in range(1, 120):
        time.sleep(1)
        if os.path.exists("/tmp/cloudflared.log"):
            try:
                with open("/tmp/cloudflared.log", "r") as f:
                    log_cf = f.read()
                match_cf = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_cf)
                if match_cf:
                    public_url = match_cf.group(0)
                    print(f"\n🔥 [Dual-Tunnel Hook] BẮT ĐƯỢC CLOUDFLARE URL: {public_url}\n")
                    break
            except Exception:
                pass
        if os.path.exists("/tmp/app.log"):
            try:
                with open("/tmp/app.log", "r") as f:
                    log_app = f.read()
                match_gradio = re.search(r"https://[a-zA-Z0-9-]+\.gradio\.live", log_app)
                if match_gradio:
                    public_url = match_gradio.group(0)
                    print(f"\n🔥 [Dual-Tunnel Hook] BẮT ĐƯỢC GRADIO LIVE URL: {public_url}\n")
                    break
            except Exception:
                pass
    
    if public_url:
        notify_kernel_log("ready", "Backend GPU đã sẵn sàng và kết nối thành công!", public_url)
        def notify(endpoint):
            try:
                is_json = "api/backend-url" in endpoint
                data = json.dumps({"url": public_url}).encode("utf-8") if is_json else public_url.encode("utf-8")
                headers = {
                    "Content-Type": "application/json" if is_json else "text/plain",
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                }
                req = urllib.request.Request(endpoint, data=data, headers=headers)
                urllib.request.urlopen(req, timeout=4)
                print(f"   ✅ [Tunnel Hook] Đã đồng bộ tới: {endpoint}")
            except Exception:
                pass

        threads = [threading.Thread(target=notify, args=(url,)) for url in TARGET_WEBHOOKS]
        for t in threads:
            t.daemon = True
            t.start()
    else:
        notify_kernel_log("error", "Không tạo được đường hầm Cloudflare/Gradio!")

# Kích hoạt Hook chạy ngầm (Non-blocking)
hook_thread = threading.Thread(target=dispatch_tunnel_url, daemon=True)
hook_thread.start()

print("=" * 70)
print("🚀 SYNEXAN AI FLOOD & TRAFFIC BACKEND ĐANG KHỞI ĐỘNG TRÊN GPU TESLA T4!")
print("=" * 70)

# Khởi chạy Gradio App với đường dẫn tuyệt đối và thư mục làm việc chuẩn xác
app_py_path = os.path.join(BACKEND_DIR, "app.py") if os.path.exists(os.path.join(BACKEND_DIR, "app.py")) else "app.py"
working_cwd = BACKEND_DIR if os.path.exists(BACKEND_DIR) else os.getcwd()

with open("/tmp/app.log", "w", buffering=1) as app_log:
    app_proc = subprocess.Popen(
        [sys.executable, app_py_path],
        cwd=working_cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in app_proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        app_log.write(line)
        app_log.flush()
    ret = app_proc.wait()

# Nếu app_proc thoát (bị ngắt hoặc lỗi):
if ret != 0:
    err_txt = open("/tmp/app.log").read()[-1200:] if os.path.exists("/tmp/app.log") else f"Process exited code {ret}"
    print(f"\n❌ [CRASH DETECTED] Mã lỗi: {ret}\n{err_txt}\n")
    notify_kernel_log("error", f"Lỗi khởi chạy backend (mã {ret}): {err_txt}")
else:
    print("\n💤 Backend kết thúc bình thường. Duy trì tiến trình ngầm...")
    while True:
        time.sleep(30)
